# Relatório Projeto Final: Integration Gateway
## Gerador de PDF: execute `Run All` para gerar o relatório

> Adicione os prints de evidência em `../evidencias/` com os nomes da **Seção 3** antes de rodar.

In [1]:
import sys, subprocess
try:
    from reportlab.lib.pagesizes import A4
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'reportlab'], check=True)
    from reportlab.lib.pagesizes import A4

from pathlib import Path
from datetime import date
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor, black, white
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.platypus import (
    Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image, HRFlowable, KeepTogether, Flowable
)
from reportlab.platypus.doctemplate import BaseDocTemplate, PageTemplate, Frame
from reportlab.pdfgen import canvas as rl_canvas
print('Imports OK')

Imports OK


In [2]:
REPORTS_DIR    = Path('..').resolve()
EVIDENCIAS_DIR = REPORTS_DIR / 'evidencias'
OUTPUT_PDF     = REPORTS_DIR / 'Relatório Projeto Final Integration Gateway.pdf'

BLUE_DARK  = HexColor('#1B3F7A')
BLUE_MED   = HexColor('#2C6FBF')
BLUE_LIGHT = HexColor('#EBF4FF')
GRAY_LIGHT = HexColor('#F0F0F0')
GRAY_LINE  = HexColor('#B0C4D8')
TEXT_DARK  = HexColor('#1A1A2E')
TEXT_GRAY  = HexColor('#555566')

PAGE_W, PAGE_H = A4
MARGIN_LR  = 2.5 * cm
MARGIN_TOP = 3.8 * cm
MARGIN_BOT = 3.0 * cm
CONTENT_W  = PAGE_W - 2 * MARGIN_LR
print(f'Saída: {OUTPUT_PDF}')
print(f'Largura útil: {CONTENT_W/cm:.2f} cm')

Saída: /home/cronos-1226/Documentos/curso-docker/integration-gateway/reports/Relatório Projeto Final Integration Gateway.pdf
Largura útil: 16.00 cm


## Seção 3: Evidências

Coloque os prints em `reports/evidencias/`. Arquivos ausentes aparecem como caixa tracejada no PDF.

| Arquivo | Evidência |
|---|---|
| `ev01_compose_ps.png` | `docker compose ps`: todos healthy |
| `ev02_healthchecks.png` | `curl` nos endpoints `/health` |
| `ev03_payload_valido.png` | Resposta 201: payload válido |
| `ev04_payload_invalido.png` | Resposta 400: payload inválido |
| `ev05_auditoria_normal.png` | `/audits/summary`: fluxo normal |
| `ev06_docker_inspect.png` | `docker inspect`: hardening |
| `ev07_incidente_502.png` | Resposta 502: broken-v2 |
| `ev08_logs_broken.png` | Logs do transformer broken-v2 |
| `ev09_logs_mock.png` | Logs do internal-api-mock |
| `ev10_auditoria_incidente.png` | `/audits/summary`: FAILED |
| `ev11_rollback.png` | `docker compose up`: pós-rollback |
| `ev12_pos_rollback.png` | Validação pós-rollback |


In [3]:
EVIDENCE = {
    'ev01': 'ev01_compose_ps.png',
    'ev02': 'ev02_healthchecks.png',
    'ev03': 'ev03_payload_valido.png',
    'ev04': 'ev04_payload_invalido.png',
    'ev05': 'ev05_auditoria_normal.png',
    'ev06': 'ev06_docker_inspect.png',
    'ev07': 'ev07_incidente_502.png',
    'ev08': 'ev08_logs_broken.png',
    'ev09': 'ev09_logs_mock.png',
    'ev10': 'ev10_auditoria_incidente.png',
    'ev11': 'ev11_rollback.png',
    'ev12': 'ev12_pos_rollback.png',
}
found   = [k for k,v in EVIDENCE.items() if (EVIDENCIAS_DIR/v).exists()]
missing = [k for k,v in EVIDENCE.items() if not (EVIDENCIAS_DIR/v).exists()]
print(f'Encontradas : {found}')
print(f'Placeholder : {missing}')

Encontradas : ['ev01']
Placeholder : ['ev02', 'ev03', 'ev04', 'ev05', 'ev06', 'ev07', 'ev08', 'ev09', 'ev10', 'ev11', 'ev12']


In [4]:
class NumberedCanvas(rl_canvas.Canvas):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        num_pages = len(self._saved_page_states)
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self._draw_hf(num_pages)
            rl_canvas.Canvas.showPage(self)
        rl_canvas.Canvas.save(self)

    def _draw_hf(self, total):
        hy = PAGE_H - 1.9 * cm
        self.setFillColor(BLUE_DARK)
        self.setFont('Helvetica-Bold', 10)
        self.drawString(MARGIN_LR, hy, 'Integration Gateway')
        self.setFillColor(BLUE_MED)
        self.setFont('Helvetica', 9)
        self.drawRightString(PAGE_W - MARGIN_LR, hy,
                             'Gerenciamento Avançado de Containers')
        self.setStrokeColor(BLUE_MED)
        self.setLineWidth(0.8)
        self.line(MARGIN_LR, hy - 0.3*cm, PAGE_W - MARGIN_LR, hy - 0.3*cm)
        fy = 1.4 * cm
        self.setStrokeColor(GRAY_LINE)
        self.setLineWidth(0.5)
        self.line(MARGIN_LR, fy + 0.45*cm, PAGE_W - MARGIN_LR, fy + 0.45*cm)
        self.setFont('Helvetica', 8)
        self.setFillColor(TEXT_GRAY)
        self.drawString(MARGIN_LR, fy, 'Projeto 5  |  Turma 2026')
        self.drawCentredString(PAGE_W/2, fy,
                               f'Página {self._pageNumber} de {total}')
        self.drawRightString(PAGE_W - MARGIN_LR, fy,
                             date.today().strftime('%d/%m/%Y'))

print('Canvas OK')

Canvas OK


In [5]:
class ImagePlaceholder(Flowable):
    def __init__(self, key, label, max_h=7*cm):
        super().__init__()
        self._key   = key
        self._label = label
        self._max_h = max_h
        self._path  = EVIDENCIAS_DIR / EVIDENCE.get(key, '')

    def wrap(self, avW, avH):
        if self._path.exists():
            img = Image(str(self._path))
            f = min(CONTENT_W / img.drawWidth, self._max_h / img.drawHeight, 1.0)
            self.width  = img.drawWidth  * f
            self.height = img.drawHeight * f
        else:
            self.width  = CONTENT_W
            self.height = min(self._max_h, 4.5*cm)
        return self.width, self.height

    def draw(self):
        if self._path.exists():
            img = Image(str(self._path))
            f = min(CONTENT_W / img.drawWidth, self._max_h / img.drawHeight, 1.0)
            w = img.drawWidth  * f
            h = img.drawHeight * f
            r = min(w, h) * 0.10
            c = self.canv
            c.saveState()
            p = c.beginPath()
            p.roundRect(0, 0, w, h, r)
            c.clipPath(p, stroke=0, fill=0)
            c.drawImage(str(self._path), 0, 0, width=w, height=h, mask='auto')
            c.restoreState()
            c.setStrokeColor(GRAY_LINE)
            c.setLineWidth(0.6)
            c.roundRect(0, 0, w, h, r, stroke=1, fill=0)
        else:
            c = self.canv
            r = min(self.width, self.height) * 0.10
            c.setFillColor(BLUE_LIGHT)
            c.setStrokeColor(BLUE_MED)
            c.setDash(5, 3)
            c.setLineWidth(1)
            c.roundRect(0, 0, self.width, self.height, r, fill=1, stroke=1)
            c.setDash()
            c.setFillColor(BLUE_MED)
            c.setFont('Helvetica-Oblique', 10)
            c.drawCentredString(self.width/2, self.height/2 + 7,
                                f'[ {self._label} ]')
            c.setFont('Helvetica', 8)
            c.setFillColor(TEXT_GRAY)
            c.drawCentredString(self.width/2, self.height/2 - 8,
                                f'Arquivo esperado: {EVIDENCE.get(self._key, "")}')


class CodeBlock(Flowable):
    def __init__(self, text, fs=8.5):
        super().__init__()
        self._text = text
        self._fs   = fs
        self._pad  = 0.25 * cm

    def wrap(self, avW, avH):
        self.width  = avW
        lh = self._fs * 1.45
        self.height = len(self._text.split('\n')) * lh + 2 * self._pad
        return self.width, self.height

    def draw(self):
        c = self.canv
        c.setFillColor(GRAY_LIGHT)
        c.setStrokeColor(GRAY_LINE)
        c.setLineWidth(0.4)
        c.rect(0, 0, self.width, self.height, fill=1, stroke=1)
        c.setFillColor(TEXT_DARK)
        c.setFont('Courier', self._fs)
        lh = self._fs * 1.45
        y  = self.height - self._pad - self._fs
        for line in self._text.split('\n'):
            c.drawString(self._pad, y, line)
            y -= lh

print('Flowables OK')

Flowables OK


In [6]:
def _s(name, **kw):
    return ParagraphStyle(name, **kw)

S = {
    'title':   _s('title',   fontName='Helvetica-Bold', fontSize=22,
                  textColor=BLUE_DARK, alignment=TA_CENTER,
                  spaceAfter=0.3*cm, leading=28),
    'sub':     _s('sub',     fontName='Helvetica',      fontSize=14,
                  textColor=BLUE_MED,  alignment=TA_CENTER,
                  spaceAfter=0.2*cm, leading=20),
    'info':    _s('info',    fontName='Helvetica',      fontSize=11,
                  textColor=TEXT_GRAY, alignment=TA_CENTER,
                  spaceAfter=0.1*cm, leading=16),
    'member':  _s('member',  fontName='Helvetica',      fontSize=12,
                  textColor=TEXT_DARK, alignment=TA_CENTER,
                  spaceAfter=0.12*cm, leading=18),
    'h2':      _s('h2',      fontName='Helvetica-Bold', fontSize=14,
                  textColor=BLUE_DARK, alignment=TA_LEFT,
                  spaceBefore=0.5*cm, spaceAfter=0.2*cm, leading=20),
    'h3':      _s('h3',      fontName='Helvetica-Bold', fontSize=11,
                  textColor=BLUE_MED,  alignment=TA_LEFT,
                  spaceBefore=0.3*cm, spaceAfter=0.1*cm, leading=16),
    'body':    _s('body',    fontName='Helvetica',      fontSize=10.5,
                  textColor=TEXT_DARK, alignment=TA_JUSTIFY,
                  spaceAfter=0.15*cm, leading=16),
    'bullet':  _s('bullet',  fontName='Helvetica',      fontSize=10.5,
                  textColor=TEXT_DARK, alignment=TA_JUSTIFY,
                  leftIndent=0.5*cm,  spaceAfter=0.1*cm, leading=16),
    'caption': _s('caption', fontName='Helvetica-Oblique', fontSize=9,
                  textColor=TEXT_GRAY, alignment=TA_CENTER,
                  spaceAfter=0.15*cm, leading=13),
    'nota':    _s('nota',    fontName='Helvetica-Oblique', fontSize=9.5,
                  textColor=BLUE_DARK, alignment=TA_JUSTIFY,
                  leftIndent=0.4*cm, rightIndent=0.4*cm,
                  spaceBefore=0.1*cm, spaceAfter=0.2*cm, leading=14),
    'tc':      _s('tc',      fontName='Helvetica',      fontSize=9,
                  textColor=TEXT_DARK, alignment=TA_LEFT,
                  leading=10, wordWrap='CJK'),
    'th':      _s('th',      fontName='Helvetica-Bold', fontSize=9,
                  textColor=white,    alignment=TA_CENTER,
                  leading=10),
}

def P(t, s='body'): return Paragraph(t, S[s])
def H2(t):          return P(t, 'h2')
def H3(t):          return P(t, 'h3')
def Body(t):        return P(t, 'body')
def Bullet(t):      return Paragraph(f'• {t}', S['bullet'])
def Caption(t):     return P(t, 'caption')
def Nota(t):        return P(t, 'nota')
def Sp(v=0.3):      return Spacer(1, v * cm)
def HR():           return HRFlowable(width='100%', thickness=0.5,
                                      color=GRAY_LINE, spaceAfter=0.2*cm)

def Tbl(data, widths, fs=9):
    # Envolve células em Paragraph para quebra automática de linha.
    head_s = ParagraphStyle('_th', fontName='Helvetica-Bold', fontSize=fs,
                            textColor=white, alignment=TA_CENTER,
                            leading=fs + 1, splitLongWords=1)
    cell_s = ParagraphStyle('_tc', fontName='Helvetica',      fontSize=fs,
                            textColor=TEXT_DARK, alignment=TA_LEFT,
                            leading=fs + 1, splitLongWords=1)
    wrapped = []
    for ri, row in enumerate(data):
        wrapped.append([
            Paragraph(str(c), head_s if ri == 0 else cell_s)
            for c in row
        ])
    col_w = [CONTENT_W * w for w in widths]
    t = Table(wrapped, colWidths=col_w, repeatRows=1)
    cmds = [
        ('BACKGROUND',    (0, 0), (-1,  0),  BLUE_DARK),
        ('VALIGN',        (0, 0), (-1, -1),  'MIDDLE'),
        ('GRID',          (0, 0), (-1, -1),  0.3, GRAY_LINE),
        ('TOPPADDING',    (0, 0), (-1, -1),  5),
        ('BOTTOMPADDING', (0, 0), (-1, -1),  5),
        ('LEFTPADDING',   (0, 0), (-1, -1),  6),
        ('RIGHTPADDING',  (0, 0), (-1, -1),  6),
    ]
    for i in range(1, len(data)):
        cmds.append(('BACKGROUND', (0, i), (-1, i),
                     white if i % 2 == 1 else BLUE_LIGHT))
    t.setStyle(TableStyle(cmds))
    return t

def Ev(key, label, max_h=7*cm):
    return [
        Sp(0.15),
        Caption(f'Evidência {key[2:]}: {label}'),
        ImagePlaceholder(key, label, max_h=max_h),
        Sp(0.25),
    ]

print('Estilos OK')

Estilos OK


In [7]:
def sec_capa():
    e = []
    e.append(Sp(3.5))
    e.append(P('Relatório Projeto Final', 'title'))
    e.append(P('Integration Gateway', 'title'))
    e.append(Sp(0.3))
    e.append(HRFlowable(width='55%', thickness=2, color=BLUE_MED,
                        spaceAfter=0.3*cm, hAlign='CENTER'))
    e.append(P('Gerenciamento Avançado de Containers', 'sub'))
    e.append(Sp(1.8))
    for nome in [
        'Allef Oliveira Ramos',
        'Fernanda de Oliveira da Costa',
        'Pedro Henrique Oliveira Dias',
        'Juliana Ballin Lima',
        'Camila Felix dos Reis',
    ]:
        e.append(P(nome, 'member'))
    e.append(Sp(1.5))
    e.append(P('Projeto 5  |  Turma 2026', 'info'))
    e.append(P(date.today().strftime('%d de %B de %Y'), 'info'))
    e.append(PageBreak())
    return e

print('Capa OK')

Capa OK


In [8]:
def sec_contexto():
    e = []
    e.append(H2('1. Contexto do Problema'))
    e.append(Body(
        'Empresas com sistemas legados recebem dados externos em formatos '
        'incompatíveis com seus contratos internos. O projeto implementa uma '
        'camada intermediária que atua como ponte entre clientes externos e '
        'a API interna, garantindo validação, transformação e rastreabilidade '
        'completa das integrações.'
    ))
    e.append(H3('Responsabilidades da camada intermediária'))
    for b in [
        'Recebe pedidos externos via HTTP',
        'Valida e transforma o payload para o contrato legado',
        'Encaminha para a API interna',
        'Registra auditoria completa no banco de dados',
        'Expõe endpoints de consulta de rastreabilidade',
    ]:
        e.append(Bullet(b))
    return e

def sec_arquitetura():
    e = []
    e.append(H2('2. Arquitetura Geral'))
    e.append(Body(
        'O ambiente é composto por cinco serviços orquestrados via Docker Compose, '
        'distribuídos em três redes isoladas conforme responsabilidade.'
    ))
    arch = ('Cliente Externo\n'
            '      |\n'
            '      v\n'
            ' gateway-api :8000       (external_net + internal_net)\n'
            '      |\n'
            '      v\n'
            ' transformer :8100       (internal_net + data_net)\n'
            '    /         \\\n'
            '   v           v\n'
            ' internal-api-mock   postgres :5432\n'
            '     :8200           (data_net)\n'
            ' (internal_net)\n'
            '      ^\n'
            '      |\n'
            ' audit-api :8300         (external_net + data_net)\n'
            '      |\n'
            '      v\n'
            'Cliente Externo (consulta)')
    e.append(Sp(0.2))
    e.append(CodeBlock(arch))
    e.append(Sp(0.3))
    return e

print('Seções 1-2 OK')

Seções 1-2 OK


In [9]:
def sec_servicos():
    e = []
    e.append(H2('3. Serviços'))
    e.append(Tbl([
        ['Serviço',          'Imagem',              'Porta Host', 'Função'],
        ['gateway-api',       'build local',         '8000',       'Ponto de entrada externo'],
        ['transformer',       'build local',         'interno',    'Valida, transforma e audita'],
        ['internal-api-mock', 'build local',         'interno',    'Simula o sistema legado'],
        ['postgres',          'postgres:15-alpine',  'nenhuma',    'Persistência de auditoria'],
        ['audit-api',         'build local',         '8300',       'Consulta de histórico'],
    ], [0.23, 0.26, 0.17, 0.34]))
    e.append(Sp(0.15))
    e.append(Nota('Postgres sem porta exposta: acessível apenas via data_net.'))
    return e

def sec_redes():
    e = []
    e.append(H2('4. Redes e Isolamento'))
    e.append(Tbl([
        ['Rede',         'Serviços',                                    'Finalidade'],
        ['external_net', 'gateway-api, audit-api',
         'Tráfego externo controlado'],
        ['internal_net', 'gateway-api, transformer, internal-api-mock',
         'Comunicação interna entre serviços'],
        ['data_net',     'transformer, audit-api, postgres',
         'Banco de dados isolado'],
    ], [0.22, 0.40, 0.38]))
    e.append(H3('Por que o gateway precisa de duas redes?'))
    e.append(Body(
        'O gateway-api é a fronteira do sistema: recebe requisições de fora via '
        'external_net e as repassa internamente via internal_net. '
        'Sem participar das duas redes, o serviço não conseguiria fazer a ponte '
        'entre os dois domínios.'
    ))
    e.append(H3('Por que o Postgres não está na external_net?'))
    e.append(Body(
        'Dados sensíveis de auditoria não devem ser expostos à rede externa. '
        'Apenas transformer e audit-api acessam o banco via data_net, '
        'seguindo o princípio do menor privilégio.'
    ))
    return e

def sec_volumes():
    e = []
    e.append(H2('5. Volumes e Persistência'))
    e.append(Tbl([
        ['Volume',          'Tipo',         'Finalidade'],
        ['postgres_data',   'named volume', 'Auditoria persiste entre reinicializações'],
        ['./db/init.sql',   'bind mount',   'Script de inicialização do banco'],
    ], [0.28, 0.22, 0.50]))
    return e

print('Seções 3-5 OK')

Seções 3-5 OK


In [10]:
def sec_healthchecks():
    e = []
    e.append(H2('6. Healthchecks'))
    e.append(Body(
        'Todos os serviços expõem /health monitorado pelo Docker. '
        'A configuração depends_on com condition: service_healthy garante a '
        'ordem de inicialização: o transformer só sobe após o postgres estar acessível.'
    ))
    e.append(Tbl([
        ['Serviço',         'Endpoint',     'Intervalo', 'Timeout', 'Retries'],
        ['gateway-api',       'GET /health',  '10s', '5s', '3'],
        ['transformer',       'GET /health',  '10s', '5s', '3'],
        ['internal-api-mock', 'GET /health',  '10s', '5s', '3'],
        ['audit-api',         'GET /health',  '10s', '5s', '3'],
        ['postgres',          'pg_isready',   '10s', '5s', '5'],
    ], [0.27, 0.25, 0.16, 0.16, 0.16]))
    e.append(Nota(
        'O transformer verifica a conexão com o banco em seu /health e '
        'só reporta healthy quando o postgres está acessível.'
    ))
    return e

def sec_limites():
    e = []
    e.append(H2('7. Limites de Recursos'))
    e.append(Body(
        'Configurados via deploy.resources.limits no Compose, garantindo que '
        'nenhum serviço consuma recursos além do necessário.'
    ))
    e.append(Tbl([
        ['Serviço',         'CPUs', 'Memória'],
        ['gateway-api',       '0.25', '128m'],
        ['transformer',       '0.50', '256m'],
        ['internal-api-mock', '0.25', '128m'],
        ['audit-api',         '0.25', '128m'],
        ['postgres',          '0.50', '256m'],
    ], [0.50, 0.25, 0.25], fs=10))
    return e

def sec_hardening():
    e = []
    e.append(H2('8. Hardening (Segurança)'))
    e.append(Body(
        'As configurações de hardening foram aplicadas nos quatro serviços Python, '
        'seguindo o princípio do menor privilégio.'
    ))
    e.append(Tbl([
        ['Configuração',  'Valor',                    'Finalidade'],
        ['user',          '1000:1000',               'Não roda como root'],
        ['read_only',     'true',                    'Filesystem somente leitura'],
        ['tmpfs',         '/tmp',                    'Escrita permitida apenas em /tmp'],
        ['cap_drop',      'ALL',                     'Remove todas as capabilities Linux'],
        ['security_opt',  'no-new-privileges:true',  'Impede escalada de privilégios'],
    ], [0.25, 0.30, 0.45]))
    e.append(H3('Evidência pelo docker inspect'))
    e.append(CodeBlock(
        'Memory: 268435456   (diferente de 0)\n'
        'NanoCpus: 500000000 (diferente de 0)\n'
        'ReadonlyRootfs: true\n'
        'CapDrop: ["ALL"]\n'
        'SecurityOpt: ["no-new-privileges:true"]'
    ))
    e += Ev('ev06', 'docker inspect confirmando hardening', max_h=6*cm)
    return e

print('Seções 6-8 OK')

Seções 6-8 OK


In [11]:
def sec_demo():
    e = []
    e.append(H2('9. Demo: Fluxo Normal'))
    steps = [
        ('9.1 Subir o ambiente',
         'docker compose up -d --build\ndocker compose ps  # todos healthy',
         'ev01', 'docker compose ps com todos os serviços healthy', 5*cm),
        ('9.2 Verificar healthchecks',
         'curl http://localhost:8000/health  # gateway-api: healthy\n'
         'curl http://localhost:8300/health  # audit-api: healthy',
         'ev02', 'Endpoints /health respondendo corretamente', 5*cm),
        ('9.3 Payload válido: resposta esperada 201',
         'sh scripts/send_valid_order.sh\n'
         '# transformer_version: stable-v1, status: success',
         'ev03', 'Resposta 201 com status success', 5*cm),
        ('9.4 Payload inválido: resposta esperada 400',
         'sh scripts/send_invalid_order.sh\n'
         '# error: missing required external fields',
         'ev04', 'Resposta 400 com campos ausentes', 5*cm),
        ('9.5 Consultar auditoria',
         'curl http://localhost:8300/audits/summary\n'
         '# SUCCESS: 3 | stable-v1',
         'ev05', '/audits/summary no fluxo normal', 5*cm),
    ]
    for title, cmd, key, cap, mh in steps:
        e.append(H3(title))
        e.append(CodeBlock(cmd))
        e += Ev(key, cap, max_h=mh)
    return e

print('Seção 9 OK')

Seção 9 OK


In [12]:
def sec_incidente():
    e = []
    e.append(H2('10. Incidente: transformer broken-v2'))
    e.append(Body(
        'Uma nova versão do transformer foi publicada com bug de contrato. '
        'O broken-v2 envia campos com nomes incorretos ao sistema legado, '
        'causando falha na integração.'
    ))
    e.append(H3('10.1 Divergência de contrato'))
    e.append(Tbl([
        ['Versão',    'Campos enviados',             'Campos esperados pelo legado'],
        ['broken-v2', 'order_id, items, sku, qty',
         'legacy_order_id, legacy_items, legacy_sku, legacy_qty'],
    ], [0.18, 0.38, 0.44]))
    e.append(H3('10.2 Reproduzir o incidente'))
    e.append(CodeBlock(
        'docker compose -f docker-compose.yml '
        '-f docker-compose.incident.yml up -d --build transformer\n'
        'sh scripts/send_valid_order.sh\n'
        '# 502: transformer_version: broken-v2, status: failed'
    ))
    e += Ev('ev07', 'Resposta 502 retornada pelo broken-v2', max_h=5*cm)
    return e

def sec_diagnostico():
    e = []
    e.append(H2('11. Diagnóstico pelo Log'))
    e.append(Body(
        'O diagnóstico foi feito cruzando dois logs. O internal-api-mock registrou '
        'os campos ausentes no payload recebido. O transformer registrou o '
        'status HTTP 422 retornado pelo mock.'
    ))
    e.append(H3('11.1 Log do transformer'))
    e.append(CodeBlock(
        '[transformer] starting on port 8100 version=broken-v2\n'
        '[transformer] FAILED correlation_id=demo-valid-001 internal_status=422'
    ))
    e += Ev('ev08', 'Logs do transformer broken-v2', max_h=5*cm)
    e.append(H3('11.2 Log do internal-api-mock'))
    e.append(CodeBlock(
        '[internal-api-mock] rejected\n'
        'missing=[legacy_order_id, legacy_customer_code,\n'
        '         legacy_total_items, legacy_items]\n'
        'payload={...}'
    ))
    e += Ev('ev09', 'Logs do internal-api-mock rejeitando campos', max_h=5*cm)
    e.append(H3('11.3 Confirmação via auditoria'))
    e.append(CodeBlock(
        'curl http://localhost:8300/audits/summary\n'
        '# FAILED | broken-v2: 1 registro'
    ))
    e += Ev('ev10', '/audits/summary durante o incidente', max_h=5*cm)
    e.append(Nota(
        'Causa raiz: campos renomeados sem atualizar o contrato legado, '
        'violando o contrato LEGACY_ORDER_V1.'
    ))
    return e

print('Seções 10-11 OK')

Seções 10-11 OK


In [13]:
def sec_rollback():
    e = []
    e.append(H2('12. Rollback e Correção'))
    e.append(Body(
        'O rollback consistiu em recompilar o transformer sem o override do '
        'arquivo de incidente, restaurando a versão stable-v1.'
    ))
    e.append(H3('12.1 Restaurar versão estável'))
    e.append(CodeBlock(
        'docker compose up -d --build transformer\n'
        '# versão stable-v1 volta ao ar'
    ))
    e += Ev('ev11', 'Rollback para stable-v1', max_h=5*cm)
    e.append(H3('12.2 Validar'))
    e.append(CodeBlock(
        'docker compose logs transformer | grep version=stable-v1\n'
        'sh scripts/send_valid_order.sh\n'
        '# 201: transformer_version: stable-v1, status: success\n\n'
        'curl http://localhost:8300/audits/summary\n'
        '# SUCCESS | stable-v1: 4 registros\n'
        '# FAILED  | broken-v2: 1 registro  (histórico imutável)'
    ))
    e += Ev('ev12', 'Validação pós-rollback', max_h=5*cm)
    e.append(Nota(
        'Os registros de falha não foram corrigidos. A auditoria é imutável '
        'e serve como prova histórica do ocorrido.'
    ))
    return e

def sec_auditoria():
    e = []
    e.append(H2('13. Como a Auditoria Ajudou'))
    e.append(Body(
        'O endpoint /audits/summary agrupou registros por status e '
        'transformer_version, tornando o diagnóstico imediato.'
    ))
    e.append(Tbl([
        ['status',  'transformer_version', 'total'],
        ['SUCCESS', 'stable-v1',           '4'],
        ['FAILED',  'broken-v2',           '1'],
        ['FAILED',  'stable-v1',           '1'],
    ], [0.33, 0.42, 0.25], fs=10))
    e.append(Sp(0.2))
    for ins in [
        'Quando os erros começaram: versão broken-v2',
        'Quando foram resolvidos: versão stable-v1 voltou ao ar',
        'O que falhou: contrato legado violado',
        'docker inspect confirma limites e hardening aplicados corretamente',
    ]:
        e.append(Bullet(ins))
    return e

def sec_perguntas():
    e = []
    e.append(H2('14. Respostas às Perguntas do Projeto'))
    qa = [
        ('14.1 Por que o Postgres não deve estar na rede externa?',
         'O banco armazena registros de auditoria com dados sensíveis das '
         'integrações. Expô-lo à rede externa criaria um vetor de ataque '
         'direto ao dado persistido. O isolamento via data_net garante que '
         'somente os serviços autorizados (transformer e audit-api) consigam '
         'acessar o banco, em conformidade com o princípio do menor privilégio.'),

        ('14.2 Por que o Gateway precisa de duas redes?',
         'O gateway-api é o único ponto de contato entre o mundo externo e a '
         'infraestrutura interna. Para cumprir esse papel ele precisa estar em '
         'external_net (receber requisições externas) e em internal_net '
         '(encaminhá-las ao transformer). Sem participar das duas redes, o '
         'serviço não conseguiria fazer a ponte entre os domínios.'),

        ('14.3 Qual serviço conhece o contrato legado?',
         'O transformer é o único serviço que conhece o contrato LEGACY_ORDER_V1. '
         'Ele traduz os campos externos (order_id, customer_code, total_items, items) '
         'para os campos esperados pelo sistema legado (legacy_order_id, '
         'legacy_customer_code, etc.). Qualquer mudança de contrato deve ser '
         'implementada e testada exclusivamente nele antes do deploy.'),

        ('14.4 Como a falha foi identificada pelos logs?',
         'O diagnóstico foi feito cruzando dois logs: o internal-api-mock '
         'registrou exatamente os campos ausentes no payload, indicando o que '
         'o transformer deveria ter enviado. O transformer, por sua vez, '
         'registrou o status HTTP 422 retornado pelo mock. Com o campo '
         'transformer_version presente em cada registro de auditoria, foi '
         'possível correlacionar imediatamente as falhas à versão broken-v2.'),

        ('14.5 O rollback corrigiu as integrações já registradas como falha?',
         'Não. Os registros de falha gerados pela versão broken-v2 permanecem '
         'inalterados na base de auditoria. Isso é intencional: a auditoria '
         'serve como prova histórica imutável do que aconteceu, permitindo '
         'rastrear quando os erros começaram e terminaram. O rollback corrige '
         'apenas as integrações processadas a partir do momento em que o '
         'stable-v1 voltou ao ar.'),

        ('14.6 Qual a importância de versionar contratos?',
         'A presença do campo transformer_version em cada registro de auditoria '
         'foi decisiva para o diagnóstico. Sem esse campo, seria impossível '
         'determinar se as falhas foram causadas por uma mudança de versão do '
         'transformer, por dados inválidos dos clientes ou por instabilidade do '
         'sistema legado. O versionamento transforma a auditoria de um simples '
         'log de erros em uma ferramenta de rastreabilidade precisa.'),
    ]
    for q, a in qa:
        e.append(H3(q))
        e.append(Body(a))
    return e

def sec_aprendizados():
    e = []
    e.append(H2('15. Aprendizados Principais'))
    for p in [
        'Isolamento de redes por responsabilidade evita exposição desnecessária de serviços',
        'depends_on com healthcheck garante inicialização segura e ordenada dos serviços',
        'Auditoria com transformer_version é fundamental para rastrear incidentes com precisão',
        'Hardening (read_only, cap_drop, no-new-privileges) é configurável no Compose sem alterar o código',
        'Rollback rápido é possível pois o docker-compose.incident.yml sobrescreve apenas o build do transformer',
        'Logs estruturados ([serviço] campo=valor) permitem diagnóstico sem acesso direto ao banco',
    ]:
        e.append(Bullet(p))
    return e

print('Seções 12-15 OK')

Seções 12-15 OK


## Seção 8: Gerar o PDF

Execute a célula abaixo. O arquivo será salvo em `reports/`.

In [14]:
def gerar_pdf():
    story = []
    story += sec_capa()
    story += sec_contexto()
    story += sec_arquitetura()
    story += sec_servicos()
    story += sec_redes()
    story += sec_volumes()
    story += sec_healthchecks()
    story += sec_limites()
    story += sec_hardening()
    story += [PageBreak()]
    story += sec_demo()
    story += [PageBreak()]
    story += sec_incidente()
    story += sec_diagnostico()
    story += [PageBreak()]
    story += sec_rollback()
    story += sec_auditoria()
    story += sec_perguntas()
    story += sec_aprendizados()

    frame = Frame(
        MARGIN_LR, MARGIN_BOT,
        CONTENT_W, PAGE_H - MARGIN_TOP - MARGIN_BOT,
        id='main', leftPadding=0, rightPadding=0,
        topPadding=0, bottomPadding=0,
    )
    doc = BaseDocTemplate(
        str(OUTPUT_PDF),
        pagesize=A4,
        pageTemplates=[PageTemplate(id='main', frames=[frame])],
        leftMargin=MARGIN_LR, rightMargin=MARGIN_LR,
        topMargin=MARGIN_TOP, bottomMargin=MARGIN_BOT,
    )
    doc.build(story, canvasmaker=NumberedCanvas)
    sz = OUTPUT_PDF.stat().st_size / 1024
    print(f'PDF gerado com sucesso!')
    print(f'Caminho : {OUTPUT_PDF}')
    print(f'Tamanho : {sz:.1f} KB')

gerar_pdf()

PDF gerado com sucesso!
Caminho : /home/cronos-1226/Documentos/curso-docker/integration-gateway/reports/Relatório Projeto Final Integration Gateway.pdf
Tamanho : 56.5 KB
